In [ ]:
try:
  import google.colab
  IN_COLAB = True
except ImportError:
  IN_COLAB = False

if IN_COLAB:
  !pip install -q pillow-heif

import pandas as pd
import numpy as np
import seaborn as sns
import requests
import os
import warnings

from matplotlib import pyplot as plt
from plotly import express as px
from PIL import Image
from bs4 import BeautifulSoup
from scipy.stats import ttest_rel, ttest_ind
from collections import OrderedDict

with warnings.catch_warnings():
  warnings.simplefilter('ignore')
  from statsmodels.stats.anova import AnovaRM

try:
  from pillow_heif import register_heif_opener
  register_heif_opener()
except ImportError:
  pass

In [ ]:
def get_data(id, mapper=None, columns=None):
  url = 'https://docs.google.com/spreadsheets/d/{0}/gviz/tq?tqx=out:csv&sheet={1}'.format(
      id,
      '0')

  data = pd.read_csv(url)
  if 'Timestamp' in data.columns:
    data = data.drop('Timestamp', axis=1)

  if mapper is not None:
    data = data.rename(mapper, axis=1)
  elif columns is not None:
    # Handle column count mismatches (e.g., if the form added new columns)
    if len(columns) < len(data.columns):
      data = data.iloc[:, :len(columns)]
    data.columns = columns

  return data.sort_values(by='group')

# Download raw data

In [5]:
# original pictures and instructions
id = '1Qpj4M3xhDO__6fIp6_Yqk9OR8Yrddk18UeXJcqgbQuw'
columns = ['group', 'drawing url', 'title', 'instructions']
drawings = get_data(id, columns=columns)
drawings

,group,drawing url,title,instructions
1,A,https://drive.google.com/open?id=1i_q4IkT0284r...,Rocketship,https://drive.google.com/open?id=1LWe9bzSBASW2...
2,B,https://drive.google.com/open?id=1AngYHmchVXKv...,Bingus the Cat,https://drive.google.com/open?id=1buVILYNgEcz3...
0,C,https://drive.google.com/open?id=1X2KgMHbYT7tf...,FOXY,https://drive.google.com/open?id=1u5mQd4QFHbSU...
3,D,https://drive.google.com/open?id=19DDLczjFIQxm...,Space Cat,https://drive.google.com/open?id=15DFeoP2MzKyn...


In [6]:
# drawing reproductions
id = '1fjCyZ3Mfuw_iy4Y-yG5N1pcSH8Z-D8movhPBk0f6Esw'
columns = ['group',
           'A repro', 'A title', 'A assumptions',
           'B repro', 'B title', 'B assumptions',
           'C repro', 'C title', 'C assumptions',
           'D repro', 'D title', 'D assumptions']
reproductions = get_data(id, columns=columns)
reproductions

ValueError: Length mismatch: Expected axis has 19 elements, new values have 13 elements

In [ ]:
# ratings
id = '1kwYA6_Z81FwA2oLkBVrCtjnFO5v15UNNVeZF5dXYX6w'
columns = ['group',
           'eval spreadsheet',
           'assumptions spreadsheet',
           'appearance A', 'appearance B', 'appearance C', 'appearance D',
           'meaning A', 'meaning B', 'meaning C', 'meaning D',
           'clarity A', 'clarity B', 'clarity C', 'clarity D',
           'efficiency A', 'efficiency B', 'efficiency C', 'efficiency D']
ratings = get_data(id, columns=columns)
ratings

# Compare original vs. reproduced drawings

In [ ]:
def get_fname(url):
  id = url[str.find(url, '?id=')+4:]
  response = requests.get(url, stream=True)
  lines = [line.decode() for line in response.iter_lines()]
  soup = BeautifulSoup('\n'.join(lines), 'html.parser')
  return soup.find_all('title')[0].string.replace(' - Google Drive', '')

def get_image(url, height=250):
  id = url[str.find(url, '?id=')+4:]
  fname = get_fname(url)
  tmp, ext = os.path.splitext(fname)
  view_url = f'https://drive.google.com/uc?export=view&id={id}'

  if not os.path.exists(fname):
    response = requests.get(view_url, stream=True)
    with open(fname, 'wb+') as f:
      for chunk in response.iter_content(1024):
        if not chunk:
          break
        f.write(chunk)

  # PIL handles HEIC natively when pillow-heif is registered
  x = Image.open(fname).convert('RGB')
  return x.resize((int(x.size[0] * height / x.size[1]), height))

def stack_images(urls, height=250, bwidth=10):
  def im2array(i):
    return np.array(i.getdata()).reshape((i.size[1], i.size[0],
                                          np.array(i.getdata()).shape[1]))

  if type(urls) is str:
    return im2array(get_image(urls, height=height))

  # otherwise, assume we have a list
  border = np.zeros((height, bwidth, 3), dtype=int)

  x = im2array(get_image(urls[0], height=height))
  for i in range(1, len(urls)):
    next = im2array(get_image(urls[i], height=height))
    x = np.concatenate([x, border, next], axis=1)

  return x

def display_image_set(urls, height=250, bwidth=10, figsize=10, ax=None):
  x = stack_images(urls, height=height, bwidth=bwidth)

  if ax is None:
    fig = plt.figure(figsize=(figsize, figsize * (x.shape[1] / x.shape[0])))
    ax = plt.gca()
  ax.imshow(x)
  ax.set_xticks([])
  ax.set_yticks([])
  return plt.gcf()

## Group A: original vs. reproduced

Reproduced images are shown in order (groups A, B, C, then D)

In [5]:
def compare_drawings(group):
  # urls
  orig = drawings.query(f'group == "{group}"')['drawing url'].values[0]
  repos = reproductions[f'{group} repro'].values

  display_image_set(orig, height=200)
  plt.title(f'Group {group} original', fontsize=14)

  display_image_set(repos, height=150)
  plt.title(f'Reproductions of group {group}\'s drawing', fontsize=14)

In [6]:
compare_drawings('A')

NameError: name 'drawings' is not defined

## Group B: original vs. reproduced

In [ ]:
compare_drawings('B')

## Group C: original vs. reproduced

In [ ]:
compare_drawings('C')

## Group D: original vs. reproduced

In [ ]:
compare_drawings('D')

# Evaluate quality of instructions

Download each group's spreadsheets and examine (a) which assumptions were made about each set of instructions, and (b) how each set of instructions was rated by different groups.

In [ ]:
def get_id(url):
  if 'docs.google.com' in url:
    start_id = '/spreadsheets/d/'
    end_id = '/edit?usp='
    return url[str.find(url, start_id)+len(start_id):str.find(url, end_id)]
  else:
    return url[str.find(url, '?id=')+4:]

In [ ]:
def get_ratings_df(url):
  df = get_data(get_id(url))
  df.columns = ['group', *[x for x in range(df.shape[1] - 1)]]
  return df

In [ ]:
# download ratings data
ratings_data = [get_ratings_df(r) for r in ratings['eval spreadsheet'].values]

In [ ]:
def get_assumptions_df(url):
  df = get_data(get_id(url), {'Group': 'group'}).iloc[:, :3]
  df.columns = ['group', 'total', 'proportion']
  return df

In [ ]:
assumptions_data = [get_assumptions_df(a) for a in ratings['assumptions spreadsheet'].values]

Sasha: here's my suggested analysis

In [ ]:
groups = 'ABCD'
for i, df in enumerate(assumptions_data):
  assumptions_data[i] = df.assign(rater=groups[i])

compiled_ratings = pd.concat(assumptions_data, axis=0).rename({'group': 'rated', 'total': 'count'}, axis=1)
compiled_ratings

In [ ]:
# compare group B ratings to all other groups
def compare_group_to_others(ratings, reference, column='count'):
  ref = ratings.query('rated == @reference')[column].values
  other = ratings.query('rated != @reference')[column].values
  result = ttest_ind(ref, other)
  df_val = len(ref) + len(other) - 2
  print(f'{column.capitalize()}, {reference} vs. others: t({df_val}) = {result.statistic:0.3f}, p = {result.pvalue:0.4f}')

# compare totals (B vs. other)
compare_group_to_others(compiled_ratings, 'B', column='count')
compare_group_to_others(compiled_ratings, 'B', column='proportion')

## For each group, plot the proportion of other groups who followed each of their instructions

In [ ]:
def prop_plot(df, name='A', color='k', alpha=0.1, ax=None):
  data = df.set_index('group').values
  x = np.arange(data.shape[1])
  y = data.mean(axis=0)

  # standard error of the mean confidence intervals
  ci = data.std(axis=0) / np.sqrt(data.shape[0] - 1)

  if ax is None:
    ax = plt.gca()

  ax.fill_between(x, y - ci, y + ci, alpha=alpha, color=color)
  ax.plot(x, y, color=color, label=name)

  ax.set_xlabel('Instruction number', fontsize=14)
  ax.set_ylabel('Proportion followed', fontsize=14)

  return ax

In [ ]:
colors = ['#00A651', '#00AEEF', '#2E3192', '#EC008C']
n_groups = len(ratings_data)
[prop_plot(ratings_data[i], color=colors[i % len(colors)], name=groups[i]) for i in range(n_groups)];
plt.legend();

## Plot the total proportion of each group's instructions that were followed by other groups, on average

In [ ]:
def instruction_barplot(dfs, groups):
  means = [np.mean(r.set_index('group').values) for r in dfs]
  plt.bar(x=list(groups), height=means, color='k')
  plt.xlabel('Group', fontsize=14)
  plt.ylabel('Average proportion of\ninstructions followed by\nother groups',
             fontsize=14)
  return plt.gcf()

In [ ]:
instruction_barplot(ratings_data, groups[:len(ratings_data)]);

## How "good" was each group at *following* instructions?

In [ ]:
def follow_barplot(dfs, groups):
  dfs = [d.set_index('group') for d in dfs]
  group_means = {g: [] for g in groups}
  for g in groups:
    for d in dfs:
      if g in d.index:
        row = d.loc[g]
        if isinstance(row, pd.DataFrame):
          vals = row.apply(pd.to_numeric, errors='coerce').values.flatten()
        else:
          vals = pd.to_numeric(row, errors='coerce').values.flatten()
        group_means[g].append(np.nanmean(vals))

  means = [np.nanmean(group_means[g]) if group_means[g] else 0 for g in groups]
  plt.bar(x=list(groups), height=means, color='k')
  plt.xlabel('Group', fontsize=14)
  plt.ylabel('Average proportion of\ninstructions followed',
             fontsize=14)
  return plt.gcf()

In [ ]:
follow_barplot(ratings_data, groups[:len(ratings_data)]);

## How many assumptions were made about each group's instructions (by other groups)?

In [ ]:
def assumptions_barplot(dfs, groups, column='total'):
  means = [dfs[i][column].mean() for i in range(len(groups))]
  plt.bar(x=list(groups), height=means, color='k')
  plt.xlabel('Group', fontsize=14)
  plt.ylabel(f'Average assumptions made\n({column})',
             fontsize=14)
  return plt.gcf()

In [ ]:
assumptions_barplot(assumptions_data, groups[:len(assumptions_data)], column='total');

## How many (normalized) assumptions were made about each group's instructions (by other groups)?

In [ ]:
assumptions_barplot(assumptions_data, groups[:len(assumptions_data)], column='proportion');

# Examining ratings of each group's instructions

In [ ]:
# reorganize the data:
#  - 1 row per rating per group
#  - column 1: group *doing* the rating
#  - column 2: group *being rated*
#  - column 3: rating category
#  - column 4: rating value

melted_ratings = ratings.drop(['eval spreadsheet', 'assumptions spreadsheet'],
                              axis=1).melt(id_vars='group',
                                           var_name='rating category',
                                           value_name='Rating')
melted_ratings.rename({'group': 'Rating group'}, axis=1, inplace=True)
melted_ratings['Rated group'] = melted_ratings['rating category'].apply(lambda x: x[-1])
melted_ratings['Rating category'] = melted_ratings['rating category'].apply(lambda x: x[:-2])
melted_ratings

In [ ]:
px.bar(data_frame=melted_ratings, x='Rated group', y='Rating', color='Rating group',
       barmode='stack', facet_col='Rating category', title='Sums of ratings across groups')

# $t$-tests between ratings for different pairs of groups

Note 1: $t$-tests generally have very low statistical power when the sample sizes are small (as in this setting).  It's not that tests are "invalid" when you run them on data with small sample sizes-- rather, any differences have to be much larger in order to detect them than comparably sized differences with larger sample sizes.  A good discussion of this issue may be found [here](https://stats.stackexchange.com/questions/37993/is-there-a-minimum-sample-size-required-for-the-t-test-to-be-valid).

Note 2: running pairwise $t$-tests *can* be a reasonable approach, but it's important to note that we're not correcting for multiple comparisons. Each time we run a statistical test, there's some probability of getting a Type I error (i.e., a false positive).  When we run multiple tests, the Type I error rate typically increases (since we could get a false positive with *any* test).  There are several ways of taking multiple comparisons into consideration in this scenario (additional discussion may be found [here](https://statistics.laerd.com/statistical-guides/one-way-anova-statistical-guide-2.php)):
  - Adjust our threshold for what we consider to be "statistically significant".  Using a more conservative threshold for individual tests reduces the Type I error rate.
  - Use a different test (e.g. an ANOVA), where many "comparisons" can effectively be performed in a single "step"

In [ ]:
# compare ratings between pairs of groups, for each category
# positive ratings mean the first group has higher ratings than the second group
categories = ['appearance', 'meaning', 'clarity', 'efficiency']
for c in categories:
  print(f'Ratings category: {c}')
  for i, g1 in enumerate(groups):
    r1 = ratings[f'{c} {g1}'].values
    for g2 in groups[(i + 1):]:
      r2 = ratings[f'{c} {g2}'].values
      t, p = ttest_rel(r1, r2)
      df = len(r1) - 1
      print(f'\t{g1} vs. {g2}: t({df}) = {t:0.3f}, p = {p:0.4f}')


## ANOVAs comparing ratings between different groups (within a category)

In [ ]:
mapper = {'Rating group': 'rating_group', 'Rated group': 'rated_group',
          'Rating category': 'category'}
df = melted_ratings.rename(mapper, axis=1)
for c in categories:
  next_data = df.query(f'category == "{c}"')
  x = AnovaRM(next_data, depvar='Rating', subject='rating_group',
              within=['rated_group'])
  results = x.fit().anova_table
  print(f'{c.capitalize()}: F({results["Num DF"].values[0]:0.0f}, {results["Den DF"].values[0]:0.0f}) = {results["F Value"].values[0]:0.3f}, p = {results["Pr > F"].values[0]:0.4f}')

# Closing thoughts

What do you think makes a "good" methods section?  Is it one that other people *think* is clear?  Or one that results in outcomes similar to the intended outcome?  Or one that spells out every step in a way that requires minimal assumptions (and therefore potential for misinterpretation)?

The analyses outlined in this notebook are far from comprehensive.  Some other ideas could include:
  - Comparing (via statistical tests) across different groups
  - Examining the text of the instructions to see if particular wordings stand out as especially effective or ineffective
  - Examining the specific aspects of the drawings that did vs. didn't match the intended originals
  - Examining "self" vs. "other" ratings-- for example:
    - Were there any groups who rated themselves much differently than other groups rated them (in a given category)?
    - Were there any categories that tended to show high levels of mismatch between how grops rated themselves vs. how others rated them?
  - How did drawing "complexity" (e.g., visual appearance, number of instructions, word counts, etc.) relate to how accurately the instructions were followed and/or how well they were received?
  - Were some drawings more (or less) "forgiving" of inaccuracies in how the instructions were followed?  E.g., if a particular instruction wasn't followed correctly, were there some drawings that would still come out basically correctly?  And/or were some drawings more "robust" to errors (e.g., maybe the received a similar label across groups despite differences in precise appearance)?

Try playing around with the data a bit and see if anything interesting (or unexpected!) pops out!